# *Data Describtion*



In [1]:
import pandas as pd
from autoimpute.imputations import SingleImputer


In [2]:
df=pd.read_csv(r"Telco_Churn_Data.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34.0,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7093 entries, 0 to 7092
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7093 non-null   object 
 1   gender            7093 non-null   object 
 2   SeniorCitizen     7093 non-null   int64  
 3   Partner           7093 non-null   object 
 4   Dependents        7093 non-null   object 
 5   tenure            7073 non-null   float64
 6   PhoneService      6940 non-null   object 
 7   MultipleLines     6927 non-null   object 
 8   InternetService   7093 non-null   object 
 9   OnlineSecurity    6951 non-null   object 
 10  OnlineBackup      6949 non-null   object 
 11  DeviceProtection  7093 non-null   object 
 12  TechSupport       6940 non-null   object 
 13  StreamingTV       7093 non-null   object 
 14  StreamingMovies   7093 non-null   object 
 15  Contract          7078 non-null   object 
 16  PaperlessBilling  6957 non-null   object 


# Data Cleaning


In [4]:
# remove CustomerID column
df.drop('customerID', axis=1, inplace=True) # because it is not useful for prediction

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality_report = pd.DataFrame({
'Missing Count' : missing,
'Missing %'     : missing_pct
}).query('`Missing Count` > 0')

print("Missing Values:")
print(quality_report.to_string())
print(f"Duplicate rows: {df.duplicated().sum()}")

Missing Values:
                  Missing Count  Missing %
tenure                       20       0.28
PhoneService                153       2.16
MultipleLines               166       2.34
OnlineSecurity              142       2.00
OnlineBackup                144       2.03
TechSupport                 153       2.16
Contract                     15       0.21
PaperlessBilling            136       1.92
PaymentMethod               178       2.51
MonthlyCharges              198       2.79
TotalCharges                151       2.13
Duplicate rows: 56


## *Remove duplicated values*

In [5]:
before = len(df)
df.drop_duplicates(inplace=True)
after  = len(df)

print(f"Rows before : {before}")
print(f"Rows removed: {before - after}")
print(f"Rows after  : {after}")

Rows before : 7093
Rows removed: 56
Rows after  : 7037


## *convert type of TotalCharges*
 


In [6]:
# type corrections
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")


gender: ['Female' 'Male'] unique values
SeniorCitizen: [0 1] unique values
Partner: ['Yes' 'No'] unique values
Dependents: ['No' 'Yes'] unique values
tenure: [ 1. 34.  2. 45.  8. 22. 10. 28. 62. 13. 16. 58. 49. 25. 69. 52. 71. 21.
 12. 30. 47. 72. 17. 27.  5. 46. 11. 70. 63. 43. 15. 60. 18. 66.  9.  3.
 31. 50. 64. 56.  7. 42. 35. 48. 29. 65. 38. 68. 32. 55. 37. 36. 41.  6.
  4. 33. 67. 23. 57. 61. 14. 20. 53. nan 40. 59. 24. 44. 19. 54. 51. 26.
  0. 39.] unique values
PhoneService: ['No' 'Yes' nan] unique values
MultipleLines: ['No phone service' 'No' 'Yes' nan] unique values
InternetService: ['DSL' 'Fiber optic' 'No'] unique values
OnlineSecurity: ['No' 'Yes' 'No internet service' nan] unique values
OnlineBackup: ['Yes' nan 'No' 'No internet service'] unique values
DeviceProtection: ['No' 'Yes' 'No internet service'] unique values
TechSupport: ['No' 'Yes' 'No internet service' nan] unique values
StreamingTV: ['No' 'Yes' 'No internet service'] unique values
StreamingMovies: ['No' 'Yes

## *Replace inconsistent values in some columns*


In [7]:
df.replace('No internet service', 'No', inplace=True)
df['PaymentMethod'] = df['PaymentMethod'].str.replace(r'\s*\(automatic\)', '', regex=True)
df['Contract'] = df['Contract'].replace('Month-to-month', 'Monthly')
df['MultipleLines']=df['MultipleLines'].replace('No phone service', 'No')
# print values of each column
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")

gender: ['Female' 'Male'] unique values
SeniorCitizen: [0 1] unique values
Partner: ['Yes' 'No'] unique values
Dependents: ['No' 'Yes'] unique values
tenure: [ 1. 34.  2. 45.  8. 22. 10. 28. 62. 13. 16. 58. 49. 25. 69. 52. 71. 21.
 12. 30. 47. 72. 17. 27.  5. 46. 11. 70. 63. 43. 15. 60. 18. 66.  9.  3.
 31. 50. 64. 56.  7. 42. 35. 48. 29. 65. 38. 68. 32. 55. 37. 36. 41.  6.
  4. 33. 67. 23. 57. 61. 14. 20. 53. nan 40. 59. 24. 44. 19. 54. 51. 26.
  0. 39.] unique values
PhoneService: ['No' 'Yes' nan] unique values
MultipleLines: ['No' 'Yes' nan] unique values
InternetService: ['DSL' 'Fiber optic' 'No'] unique values
OnlineSecurity: ['No' 'Yes' nan] unique values
OnlineBackup: ['Yes' nan 'No'] unique values
DeviceProtection: ['No' 'Yes'] unique values
TechSupport: ['No' 'Yes' nan] unique values
StreamingTV: ['No' 'Yes'] unique values
StreamingMovies: ['No' 'Yes'] unique values
Contract: ['Monthly' 'One year' 'Two year' nan] unique values
PaperlessBilling: ['Yes' 'No' nan] unique values
P

## *Handle missing values*

In [8]:
missing_columns_category = [col for col in quality_report.index if df[col].dtype == 'object']
# hanle missing values
strategy_dict = {col: 'mode' for col in missing_columns_category}
df[missing_columns_category] = SingleImputer(
    strategy=strategy_dict,
    seed=42
).fit_transform(df[missing_columns_category])

# fill missing values in numerical columns using median per group
df['MonthlyCharges'] = df.groupby('InternetService')['MonthlyCharges'].transform(lambda x: x.fillna(x.median()))
df['tenure'] = df.groupby('Contract')['tenure'].transform(lambda x: x.fillna(x.median()))
df['tenure'] = df['tenure'].round().astype(int)
#spacial case fot total charges because it is related to tenure and monthley charges (if tenure is 0 then total charges eqal monthley charges)
mask_tc = df['TotalCharges'].isnull() & (df['tenure'] == 0)
df.loc[mask_tc, 'TotalCharges'] = df.loc[mask_tc, 'MonthlyCharges']

df['TotalCharges'] = df['TotalCharges'].fillna(
    df['tenure'] * df['MonthlyCharges'])

print("Number of missing values:\n", df.isnull().sum())
df.head()


Number of missing values:
 gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Monthly,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Monthly,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer,42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Monthly,Yes,Electronic check,70.70,151.65,Yes


In [9]:
pivot = pd.pivot_table(
df,
values   = 'MonthlyCharges',
index    = 'Contract',
columns  = 'InternetService',
aggfunc  = 'mean'
).round(2)

print("Average Monthly Charges by Contract x Internet Service:")
print(pivot.head())

# Churn Rate per Contract type
churn_pivot = pd.pivot_table(
df,
values  = 'Churn',
index   = 'Contract',
aggfunc = lambda x: (x == 'Yes').sum() / len(x) * 100
).round(2)
churn_pivot.columns = ['Churn Rate %']
print(r"\nChurn Rate % by Contract Type:")
print(churn_pivot.head())

Average Monthly Charges by Contract x Internet Service:
InternetService    DSL  Fiber optic     No
Contract                                  
Monthly          50.51        87.26  20.39
One year         61.14        98.64  20.82
Two year         70.16       104.19  21.71
\nChurn Rate % by Contract Type:
          Churn Rate %
Contract              
Monthly          42.64
One year         11.10
Two year          2.83


# *Encoding*

In [10]:
# Binary
binary_cols = [col for col in df.columns if df[col].nunique() == 2 and df[col].dtype == 'object']
for col in binary_cols:
    df[col] = df[col].map({'Yes':1, 'No':0, 'Male':1, 'Female':0})
# Ordinal data
df['Contract'] = df['Contract'].map({
    'Monthly': 0,
    'One year': 1,
    'Two year': 2
})

# One Hot
df= pd.get_dummies(df, columns=[
    'InternetService',
    'PaymentMethod',
    'MultipleLines'
],dtype=int)


df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,Churn,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer,PaymentMethod_Credit card,PaymentMethod_Electronic check,PaymentMethod_Mailed check,MultipleLines_0,MultipleLines_1
0,0,0,1,0,1,0,0,1,0,0,...,0,1,0,0,0,0,1,0,1,0
1,1,0,0,0,34,1,1,0,1,0,...,0,1,0,0,0,0,0,1,1,0
2,1,0,0,0,2,1,1,1,0,0,...,1,1,0,0,0,0,0,1,1,0
3,1,0,0,0,45,0,1,0,1,1,...,0,1,0,0,1,0,0,0,1,0
4,0,0,0,0,2,1,0,0,0,0,...,1,0,1,0,0,0,1,0,1,0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7037 entries, 0 to 7091
Data columns (total 26 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   gender                          7037 non-null   int64  
 1   SeniorCitizen                   7037 non-null   int64  
 2   Partner                         7037 non-null   int64  
 3   Dependents                      7037 non-null   int64  
 4   tenure                          7037 non-null   int64  
 5   PhoneService                    7037 non-null   int64  
 6   OnlineSecurity                  7037 non-null   int64  
 7   OnlineBackup                    7037 non-null   int64  
 8   DeviceProtection                7037 non-null   int64  
 9   TechSupport                     7037 non-null   int64  
 10  StreamingTV                     7037 non-null   int64  
 11  StreamingMovies                 7037 non-null   int64  
 12  Contract                        7037 no

In [12]:
# print values of each column
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")

gender: [0 1] unique values
SeniorCitizen: [0 1] unique values
Partner: [1 0] unique values
Dependents: [0 1] unique values
tenure: [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39] unique values
PhoneService: [0 1] unique values
OnlineSecurity: [0 1] unique values
OnlineBackup: [1 0] unique values
DeviceProtection: [0 1] unique values
TechSupport: [0 1] unique values
StreamingTV: [0 1] unique values
StreamingMovies: [0 1] unique values
Contract: [0 1 2] unique values
PaperlessBilling: [1 0] unique values
MonthlyCharges: [29.85 56.95 53.85 ... 63.1  44.2  78.7 ] unique values
TotalCharges: [  29.85 1889.5   108.15 ...  346.45  306.6  6844.5 ] unique values
Churn: [0 1] unique values
InternetService_DSL: [1 0] unique values
InternetService_Fiber optic: [0 1] unique values
InternetService_No: [0 1] unique values
Paym